# Intro

* How to implement multiple Transformer blocks
* How to build a model out of multiple embedded Classes

**Why multiple Transformer blocks?**

Language is highly nonlinear, and having more parameters across more (deeper) layers helps the model learn complex transformations and representations

# Code

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# pytorch stuff
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Hyperparameters

In [4]:
# data hyperparameters
seq_len = 8 # aka context window
n_vocab = tokenizer.vocab_size

# model hyperparameters
embed_dim = 128
n_transformer_blocks = 12

# training hyperparameters
batch_size = 5

## One Attention Head

In [5]:
# create one attention head
class OneAttentionHead(nn.Module):
  def __init__(self, embed_dim):
    super().__init__()

    # create the k/q/v matrices
    self.key   = nn.Linear(embed_dim, embed_dim, bias = False)
    self.query = nn.Linear(embed_dim, embed_dim, bias = False)
    self.value = nn.Linear(embed_dim, embed_dim, bias = False)
    self.W0    = nn.Linear(embed_dim, embed_dim, bias = False)

  def forward(self, x):
    # run the token embeddings vectors through attention
    k = self.key(x)
    q = self.query(x)
    v = self.value(x)
    y = F.scaled_dot_product_attention(q,k,v,is_causal = True)
    y = self.W0(y) # linear weightings post-attention

    return y

# Transformer block

In [6]:
class TransformerBlock(nn.Module):
  def __init__(self, embed_dim):
    super().__init__()

    # attention sublayer
    self.layer_norm_attn = nn.LayerNorm(embed_dim)
    self.attn            = OneAttentionHead(embed_dim)

    # feedforward (MLP) sublayer
    self.layer_norm_mlp  = nn.LayerNorm(embed_dim)
    self.W1              = nn.Linear(embed_dim, 4*embed_dim)
    self.gelu            = nn.GELU()
    self.W2              = nn.Linear(4*embed_dim, embed_dim)

  def forward(self, x):

    # ------- attention sublayer ------ #
    x = x + self.attn(self.layer_norm_attn(x))
    # --------------------------------- #

    # ------- MLP sublayer ------------ #
    y = x + self.W2(self.gelu(self.W1(self.layer_norm_mlp(x))))
    # --------------------------------- #

    return y

## Full Model

In [7]:
class LanguageModel(nn.Module):
  def __init__(self, n_transformer_blocks, embed_dim):
    super().__init__()

    # embedding matrices
    self.embedding = nn.Embedding(n_vocab, embed_dim)
    self.positions = nn.Embedding(seq_len, embed_dim)

    # n multiple Transformer blocks
    self.transformer_blocks = nn.Sequential(*[TransformerBlock(embed_dim) for _ in range(n_transformer_blocks)])

    # embedding to output (linear) layer
    self.final_layer_norm = nn.LayerNorm(embed_dim) # final layernorm after transformer block
    self.final_linear     = nn.Linear(embed_dim, n_vocab, bias = False)

    # the final output layer is tied to the token embedding
    self.final_linear.weight = nn.Parameter(self.embedding.weight)

  def forward(self, tok_x):

    ## embeddings ##
    token_embed = self.embedding(tok_x)
    posit_embed = self.positions(torch.arange(tok_x.shape[-1]))
    x = token_embed + posit_embed # [batch, seq_len, embedding_dim]


    ## transformer blocks ##
    x = self.transformer_blocks(x)

    ## finally, unembeddings ##
    x = self.final_layer_norm(x)
    x = self.final_linear(x)

    return x

  def generate(self, tok_x, temperature = 1., n_new_tokens = 50):

    for _ in range(n_new_tokens):

      # get predictions, but only from the past seq_len tokens
      x = self(tok_x[:, -seq_len:]) # [batch, seq_len, n_vocab]

      # extract the final token to predict the next
      x = x[:,-1,:] # [batch, n_vocab]

      # apply softmax to get probability values over all tokens in the vocab
      probs = F.softmax(x/temperature, dim = -1) # [batch, n_vocab]

      # probabilistically sample from the distribution
      tokx_next = torch.multinomial(probs, num_samples=1) # [batch, 1]

      # append
      tok_x = torch.cat((tok_x, tok_x_next), dim = 1) # [batch, (tokens+1)]

    return tok_x

## Create a model instance and inspect

In [8]:
llm = LanguageModel(n_transformer_blocks, embed_dim)
llm

LanguageModel(
  (embedding): Embedding(50257, 128)
  (positions): Embedding(8, 128)
  (transformer_blocks): Sequential(
    (0): TransformerBlock(
      (layer_norm_attn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): OneAttentionHead(
        (key): Linear(in_features=128, out_features=128, bias=False)
        (query): Linear(in_features=128, out_features=128, bias=False)
        (value): Linear(in_features=128, out_features=128, bias=False)
        (W0): Linear(in_features=128, out_features=128, bias=False)
      )
      (layer_norm_mlp): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (W1): Linear(in_features=128, out_features=512, bias=True)
      (gelu): GELU(approximate='none')
      (W2): Linear(in_features=512, out_features=128, bias=True)
    )
    (1): TransformerBlock(
      (layer_norm_attn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attn): OneAttentionHead(
        (key): Linear(in_features=128, out_features=128, bias=F

In [9]:
llm.transformer_blocks[2]

TransformerBlock(
  (layer_norm_attn): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (attn): OneAttentionHead(
    (key): Linear(in_features=128, out_features=128, bias=False)
    (query): Linear(in_features=128, out_features=128, bias=False)
    (value): Linear(in_features=128, out_features=128, bias=False)
    (W0): Linear(in_features=128, out_features=128, bias=False)
  )
  (layer_norm_mlp): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (W1): Linear(in_features=128, out_features=512, bias=True)
  (gelu): GELU(approximate='none')
  (W2): Linear(in_features=512, out_features=128, bias=True)
)

In [10]:
llm.transformer_blocks[4].attn

OneAttentionHead(
  (key): Linear(in_features=128, out_features=128, bias=False)
  (query): Linear(in_features=128, out_features=128, bias=False)
  (value): Linear(in_features=128, out_features=128, bias=False)
  (W0): Linear(in_features=128, out_features=128, bias=False)
)

In [11]:
llm.transformer_blocks[4].attn.query.weight.detach()

tensor([[ 0.0615, -0.0042, -0.0807,  ...,  0.0731,  0.0459,  0.0129],
        [-0.0055,  0.0022, -0.0199,  ..., -0.0268, -0.0515, -0.0099],
        [ 0.0882,  0.0185, -0.0385,  ..., -0.0218,  0.0749, -0.0348],
        ...,
        [-0.0276,  0.0843, -0.0201,  ...,  0.0872,  0.0464,  0.0313],
        [ 0.0115, -0.0008, -0.0615,  ..., -0.0396, -0.0721,  0.0575],
        [-0.0275,  0.0449,  0.0775,  ...,  0.0807,  0.0831,  0.0254]])

In [14]:
# create data
tokens = tokenizer.encode('I prefer oat milk in my coffee.')
X = torch.tensor(tokens[:-1]).unsqueeze(0)
y = torch.tensor(tokens[1:]).unsqueeze(0)

print(X.shape)
print(y.shape)

torch.Size([1, 8])
torch.Size([1, 8])


In [15]:
out = llm(X)
out

tensor([[[  8.6243,  -8.8844,  11.8744,  ...,  -9.5553,   4.1387,   1.6513],
         [-19.5319,   3.1553,  13.1940,  ...,  -9.3246,  -8.8763,  19.2464],
         [-31.9820,  -0.3679,  -4.2110,  ..., -17.2262,  -6.8681,  -1.0008],
         ...,
         [  6.6844,  16.8803,  -0.0791,  ...,  -9.8839,  -5.3271,   3.6951],
         [ -5.5013,  14.3997,  11.3540,  ..., -28.3829,  11.8062,   3.1212],
         [ 24.9454,   2.3172,   2.4207,  ..., -15.1399,  -6.7150,  -1.6941]]],
       grad_fn=<UnsafeViewBackward0>)

In [16]:
out.shape

torch.Size([1, 8, 50257])